# 第4章：卷积神经网络 (CNN)

> "CNN的设计不是凭空想象，而是基于对图像的两个关键观察——每个简化都有的放矢。"

## 本章知识导图

```
卷积神经网络 (Convolutional Neural Network)
│
├── 动机：全连接处理图像的问题
│   └── 100x100图像→1000个神经元 = 3000万参数！不利用空间结构
│
├── 观察1：检测模式不需要整张图
│   └── 简化1：感受野(Receptive Field) — 每个神经元只看局部
│
├── 观察2：同样的模式可能出现在不同位置
│   └── 简化2：共享参数(Weight Sharing) — 同一检测器整图复用
│
├── 观察3：下采样不影响模式检测
│   └── 简化3：汇聚(Pooling) — 缩减特征图尺寸
│
├── 卷积 vs 全连接的完整对比
└── 应用：AlphaGo为什么也用CNN？(但去掉了Pooling！)
```

## 4.1 为什么全连接网络不够用？

### 参数爆炸

一张100×100的RGB图片（3通道），展平成30000维向量。如果第一层隐藏层有1000个神经元，连接数就是30000×1000 = **3000万个参数**。这还只是第一层！

三个问题：
1. **参数太多**：容易过拟合，需要海量数据
2. **计算量太大**：训练慢
3. **没利用图像的空间结构**：全连接把像素当成互不相关的独立变量，但实际上相邻像素高度相关

### CNN的解决思路

CNN用**三个关键观察**推导出三个简化，每一项都有明确的动机：

## 4.2 观察1 → 简化1：感受野

### 观察
识别一只"猫耳朵"，只看一个局部区域就够了，不需要看整张图片。每个神经元只负责一个小区域内的模式检测。

### 简化：感受野 (Receptive Field)
每个神经元不再连接全部输入像素，而是只连接一个**局部区域**（如3×3、5×5）。

- 3×3的感受野 → 每个神经元只有3×3×3=27个输入（RGB）
- 参数从3000万降到几万！

### 感受野的关键参数
| 参数 | 含义 | 常见选择 |
|------|------|---------|
| Kernel Size | 感受野的大小 | 3×3（最常用）、5×5、7×7 |
| Stride | 感受野滑动的步长 | 1（覆盖所有位置）、2（下采样） |
| Padding | 边缘填充 | same（输出同尺寸）、valid（不填充） |

多个卷积层堆叠 → **感受野逐层扩大**。第1层看3×3，第2层看5×5，第3层看7×7……

## 4.2.1 卷积操作详解：一步一步的数值示例

### 什么是卷积？

卷积（Convolution）本质上是**元素间乘法然后求和**的操作。一个卷积核（滤波器）在输入图像上滑动，在每个位置计算一个加权和。

### 具体数值示例：3×3卷积核在5×5输入上的操作

假设我们有一个5×5的单通道输入图像：

```
输入图像 (5×5):
┌              ┐
│ 1  2  0  1  0 │
│ 0  1  2  1  1 │
│ 3  0  1  2  0 │
│ 2  1  0  1  1 │
│ 1  0  2  0  1 │
└              ┘
```

一个3×3的卷积核（例如用于检测竖边的Sobel滤波器）：

```
卷积核 (3×3):
┌             ┐
│ -1  0  1 │
│ -2  0  2 │
│ -1  0  1 │
└             ┘
```

**第一步：将卷积核放在输入的左上角**

```
取输入(0:3, 0:3):
┌           ┐     ┌            ┐
│ 1  2  0 │     │ -1   0   1 │
│ 0  1  2 │  ⊙  │ -2   0   2 │  = 逐元素相乘再求和
│ 3  0  1 │     │ -1   0   1 │
└           ┘     └            ┘
```

计算过程：
$$\begin{align}
\text{output}[0,0] &= (1 \times -1) + (2 \times 0) + (0 \times 1) \\
&+ (0 \times -2) + (1 \times 0) + (2 \times 2) \\
&+ (3 \times -1) + (0 \times 0) + (1 \times 1) \\
&= -1 + 0 + 0 + 0 + 0 + 4 - 3 + 0 + 1 \\
&= 1
\end{align}$$

**第二步：卷积核向右滑动一格（stride=1）**

取输入(0:3, 1:4)（即第二列到第四列...但实际上，如果stride=1，应该覆盖第一行前三列后再向右移一格，即取(0:3, 1:4)）:

```
取输入(0:3, 1:4):
┌           ┐
│ 2  0  1 │
│ 1  2  1 │
│ 0  1  2 │
└           ┘
```

$$\text{output}[0,1] = (2 \times -1) + (0 \times 0) + (1 \times 1) + (1 \times -2) + (2 \times 0) + (1 \times 2) + (0 \times -1) + (1 \times 0) + (2 \times 1) = -2+0+1-2+0+2+0+0+2 = 1$$

经过完整的卷积操作（stride=1, valid padding），输出是一个3×3的特征图：

```
输出特征图 (3×3):
┌              ┐
│ 1   1  -2 │
│ 3   1  -1 │
│ -1  -3  1 │
└              ┘
```

### 卷积的物理意义

这个Sobel核检测**竖直方向的亮度变化**（竖边）。正值表示从左到右变亮，负值表示从左到右变暗。<br>
不同卷积核检测不同特征：
- 竖边检测器：`[[-1,0,1],[-2,0,2],[-1,0,1]]`
- 横边检测器：`[[-1,-2,-1],[0,0,0],[1,2,1]]`
- 模糊：`[[1/9,1/9,1/9],[1/9,1/9,1/9],[1/9,1/9,1/9]]`
- 锐化：`[[0,-1,0],[-1,5,-1],[0,-1,0]]`

> **关键洞察：** 在传统图像处理中，这些核是人工设计的。在深度学习中，CNN自动从数据中**学习**这些核的数值——这比人工设计更强大，因为模型可以学到任务特定的最优检测器！</cell>


## 4.3 观察2 → 简化2：共享参数

### 观察
检测"猫耳朵"的模式检测器，无论耳朵在左上角还是右下角都应该能用。同一种模式可能在图像的**任何位置**出现。

### 简化：权重共享 (Weight Sharing)
不同位置的感受野，使用**同一组权重**！

- 每个"模式检测器"（滤波器/filter）在整张图上滑动
- 每个位置做同样的计算（卷积操作）
- 一个3×3的滤波器在整个图像上只有9个参数（加1个bias）

### 这带来了平移等变性 (Translation Equivariance)
输入图像平移 → 输出特征图也对应平移。这是CNN最重要的归纳偏置(inductive bias)。

> **直觉：** 每个滤波器学习一种"模式"——比如一个滤波器学"竖边"，一个学"横边"，一个学"斜角"。多个滤波器组合起来就能识别复杂图案。

## 4.2.2 步长(Stride)与填充(Padding)深度剖析

### 步长(Stride)的含义

步长决定卷积核每次滑动多少个像素。Stride=1时，卷积核逐个像素移动；Stride=2时，每次跳两个像素。

**给定输入尺寸$W_{in}$，核大小$K$，步长$S$，padding $P$，输出尺寸：**
$$W_{out} = \left\lfloor \frac{W_{in} - K + 2P}{S} \right\rfloor + 1$$

各种参数组合的输出尺寸示例（输入28×28, 核3×3）：

| Stride | Padding | 输出尺寸 | 说明 |
|--------|---------|----------|------|
| 1 | 0 (valid) | 26×26 | 边缘像素被丢弃 |
| 1 | 1 (same) | 28×28 | 保持尺寸不变 |
| 2 | 0 | 13×13 | 下采样×2 |
| 2 | 1 | 14×14 | 下采样×2，保留边缘 |

### Padding详解

**Valid Padding ($P=0$):**
- 不添加任何边缘填充
- 输出尺寸严格变小
- 边缘像素参与计算次数少 → 可能丢失边缘信息

**Same Padding ($P=(K-1)/2$):**
- 在输入四周填充0（通常使用`zero padding`）
- 输出尺寸与输入相同（当$S=1$时）
- 边缘像素获得更多"上下文"

**边缘处发生了什么？**

假设输入是5×5，核3×3，stride=1：

```
Valid Padding (不填充):           Same Padding (填充一圈0):
┌─ ─ ─ ─ ─ ─ ─┐                  ┌─────────────────┐
│· · · · · · ·│                  │0  0  0  0  0  0  0│
│· 1 2 0 1 0 ·│                  │0 ·1· ·2· ·0·  1  0│  ← 角部像素被覆盖1次
│· 0 1 2 1 1 ·│                  │0 ·0·  1   2   1  1│     (参与卷积次数少)
│· 3 0 1 2 0 ·│                  │0 ·3·  0   1   2  0│
│· 2 1 0 1 1 ·│                  │0  2   1   0   1  1│
│· 1 0 2 0 1 ·│                  │0  1   0   2   0  1│
│· · · · · · ·│                  │0  0   0   0   0  0│
└─ ─ ─ ─ ─ ─ ─┘                  └─────────────────┘

输出: 3×3                          输出: 5×5
每个像素被覆盖9次                   边缘像素被覆盖4-6次，中心9次
边缘信息丢失                        边缘信息被保留
```

> **实践建议：** 现代CNN中，几乎总是使用padding='same'（PyTorch中`padding=K//2`），配合3×3卷积保持空间尺寸，下采样完全交给stride=2或MaxPooling处理。这种设计最早由VGG网络提出。</cell>


## 4.3.1 特征图可视化：CNN各层"看到"了什么？

### CNN是一个层次化特征提取器

CNN不同层学到不同抽象级别的特征，这已经被Zeller & Fergus (2014)等经典论文通过反卷积可视化证实：

```
输入图像
    ↓
Layer 1: 低级特征（边缘、颜色、纹理）
    - Gabor-like边缘检测器（不同方向和频率）
    - 颜色斑点检测器
    - 简单纹理检测器
    ↓
Layer 2: 中级特征（形状、图案）
    - 角点、曲线
    - 平行线
    - 重复纹理图案
    ↓
Layer 3: 高级特征（物体部件）
    - 眼睛、鼻子（对人脸数据）
    - 车轮、车窗（对车辆数据）
    - 几何形状组合
    ↓
Layer 4+: 语义特征（完整物体/概念）
    - 完整人脸
    - 完整汽车
    - 类别特定模式
```

### 可视化技术

1. **直接可视化卷积核（第一层）**：只有第一层的卷积核可以直接"看见"，因为它们直接作用于RGB像素。更高层的核作用于抽象特征，无法直接可视化。

2. **特征图可视化（各层输出）**：
   - 给定一个输入图像
   - 前向传播，取出某一层的输出特征图（如64通道，每个通道是一个"激活图"）
   - 观察每个通道对图像的哪些区域"响应强烈"

3. **Grad-CAM（类激活图）**：
   - 通过梯度回传，生成热力图显示图像中哪些区域对分类决策贡献最大
   - 可以直观回答"模型凭什么认为这是猫？"

> **核心洞察：** CNN不是黑箱！通过学习到的特征可以看出它是否真正理解了图像。如果模型错误地依赖背景而非物体本身，特征图会暴露这个问题。</cell>


## 4.3.2 卷积层 vs 全连接层：参数效率对比

### 参数量的数学比较

考虑一个具体任务：输入是28×28的单通道图像，我们想要输出一个14×14×32的特征图。

**方案1：用卷积层（3×3核，stride=2, padding=1）**
$$P_{\text{conv}} = K^2 \times C_{in} \times C_{out} + C_{out} = 3^2 \times 1 \times 32 + 32 = 288 + 32 = 320$$

**方案2：用等效的全连接层**
- 输入维度：28×28 = 784
- 输出维度：14×14×32 = 6,272
- 参数 = 784 × 6,272 + 6,272 = **4,917,248**

| 对比维度 | 卷积层 | 全连接层 | 比值 |
|----------|--------|----------|------|
| 参数量 | 320 | 4,917,248 | ×15,366 |
| 参数量增长率 | $O(K^2 C_{in}C_{out})$ (与空间尺寸无关！) | $O(W^2H^2C_{in}C_{out})$ (输入输出尺寸的乘积) | — |
| 平移等变性 | ✅ 天然具备 | ❌ 需要数据增强学习 | — |
| 空间泛化 | ✅ 3×3模式在28×28上都适用 | ❌ 每个位置需要单独的参数 | — |
| 参数量随输入尺寸变化 | **不变！** 224×224也是320参数 | **爆炸**：224×224→约300M | — |

### 更深层的对比

假设输入3×224×224，第一层输出64通道，核7×7：

| 层类型 | 参数 | 备注 |
|--------|------|------|
| Conv2d(3,64,7) | 3×7²×64 + 64 = 9,472 | 固定 |
| Linear(150528, 64) | 150528×64 + 64 ≈ 9.6M | 爆炸！ |

> **本质差异：** 卷积通过**参数共享**和**局部连接**强加了两个归纳偏置，大幅减少了参数。这看似是"限制"，实际上是用领域知识（图像的平移等变性和局部性）换来了**极大的参数效率**和**更好的泛化**。</cell>


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# 手动卷积：用Sobel竖边检测核对图像做卷积
# 读取一张示例图像（使用skimage或创建一个简单的测试图像）
try:
    from skimage import data, color
    img = color.rgb2gray(data.camera())  # 经典cameraman图像
    print("使用skimage的camera图像")
except:
    # 如果没有skimage，创建一个合成的测试图像
    print("没有skimage，使用合成图像")
    img = np.zeros((100, 100))
    img[30:70, 20:40] = 1.0    # 白色竖条，产生竖边
    img[30:70, 60:80] = 0.5    # 灰色竖条
    img[10:90, 45:55] = 0.8    # 白色横条

# 将图像转为tensor
img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
H, W = img_tensor.shape[2], img_tensor.shape[3]
print(f"图像尺寸: {H}×{W}")

# 定义Sobel边缘检测核
# Sobel-X (检测竖边)             Sobel-Y (检测横边)
sobel_x = torch.tensor([[-1., 0., 1.],
                         [-2., 0., 2.],
                         [-1., 0., 1.]]).unsqueeze(0).unsqueeze(0)  # (1,1,3,3)

sobel_y = torch.tensor([[-1., -2., -1.],
                         [ 0.,  0.,  0.],
                         [ 1.,  2.,  1.]]).unsqueeze(0).unsqueeze(0)  # (1,1,3,3)

# 用F.conv2d手动做卷积
edge_x = F.conv2d(img_tensor, sobel_x, padding=1)  # same padding
edge_y = F.conv2d(img_tensor, sobel_y, padding=1)
edge_magnitude = torch.sqrt(edge_x**2 + edge_y**2)  # 边缘强度

print(f"竖边检测输出: {edge_x.shape}")
print(f"横边检测输出: {edge_y.shape}")
print(f"边缘强度(梯度幅值): {edge_magnitude.shape}")

# 可视化
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_tensor.squeeze(), cmap='gray')
axes[0].set_title('原始图像')
axes[0].axis('off')

axes[1].imshow(edge_x.squeeze().detach(), cmap='seismic', vmin=-1, vmax=1)
axes[1].set_title('Sobel-X (竖边检测)')
axes[1].axis('off')

axes[2].imshow(edge_y.squeeze().detach(), cmap='seismic', vmin=-1, vmax=1)
axes[2].set_title('Sobel-Y (横边检测)')
axes[2].axis('off')

axes[3].imshow(edge_magnitude.squeeze().detach(), cmap='hot')
axes[3].set_title('边缘强度 (√(X²+Y²))')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\n=== 手动卷积的本质 ===")
print("1. 卷积 = 用一个固定大小的核(W_kernel)在输入图像上滑动")
print("2. 每个位置 = W * input_patch 的逐元素乘积之和")
print("3. CNN训练 = 从数据中学习最优的核(而非人工设计)")
print("4. F.conv2d 是PyTorch最底层的卷积函数（无参数，手动传weight）")
print("5. nn.Conv2d 是对F.conv2d的封装（自动管理可学习参数）")</cell>


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子保证可重复性
torch.manual_seed(42)

# ============================================================
# 第1部分：训练CNN on FashionMNIST
# ============================================================
print("=" * 60)
print("第1部分：在FashionMNIST上训练CNN")
print("=" * 60)

# 数据加载
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))  # FashionMNIST的均值和标准差
])

train_set = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

# FashionMNIST类别名称
class_names = ['T恤/上衣', '裤子', '套头衫', '连衣裙', '外套',
               '凉鞋', '衬衫', '运动鞋', '包', '短靴']

# 定义CNN模型（带hook用于提取特征图）
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)   # 28→14
        
        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)   # 14→7
        
        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)   # 7→3
        
        # 分类头
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# 训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FashionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"使用设备: {device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

for epoch in range(3):  # 快速训练3个epoch
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # 评估
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Epoch [{epoch+1}/3] Loss: {running_loss/len(train_loader):.4f} | "
          f"Test Accuracy: {100*correct/total:.2f}%")

print("训练完成！\n")</cell>


In [ ]:
# ============================================================
# 第2部分：可视化特征图——CNN各层"看到"了什么
# ============================================================
print("=" * 60)
print("第2部分：可视化特征图")
print("=" * 60)

# 取一张测试图像
test_iter = iter(test_loader)
images, labels = next(test_iter)
image = images[0:1].to(device)  # 取第一张图
label = labels[0].item()
print(f"选中图像标签: {class_names[label]} (类别{label})")

# 注册hook: 在前向传播时自动提取每一层的输出
activations = {}
def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

model.eval()
model.conv1.register_forward_hook(get_activation('conv1_out'))
model.conv2.register_forward_hook(get_activation('conv2_out'))
model.conv3.register_forward_hook(get_activation('conv3_out'))

# 前向传播触发hook
with torch.no_grad():
    output = model(image)
    _, pred = torch.max(output, 1)
    print(f"模型预测: {class_names[pred.item()]} (置信度: {F.softmax(output, dim=1)[0, pred].item():.3f})")

print(f"\n各层特征图统计:")
for name, act in activations.items():
    # act shape: (1, C, H, W)
    print(f"  {name}: shape={act.shape}, "
          f"min={act.min():.3f}, max={act.max():.3f}, "
          f"激活率(>0)={(act > 0).float().mean()*100:.1f}%")

# 可视化conv1的特征图（前16个通道）
conv1_maps = activations['conv1_out'][0]  # (32, 28, 28)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat[:16]):
    ax.imshow(conv1_maps[i].cpu(), cmap='viridis')
    ax.set_title(f'Channel {i}')
    ax.axis('off')
plt.suptitle(f'Conv1 特征图 (输入: {class_names[label]}) - 低级特征：边缘、纹理', fontsize=14)
plt.tight_layout()
plt.show()

# 可视化conv2的特征图（前16个通道）
conv2_maps = activations['conv2_out'][0]  # (64, 14, 14)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat[:16]):
    ax.imshow(conv2_maps[i].cpu(), cmap='viridis')
    ax.set_title(f'Channel {i}')
    ax.axis('off')
plt.suptitle(f'Conv2 特征图 - 中级特征：形状、图案', fontsize=14)
plt.tight_layout()
plt.show()

# 可视化conv3的特征图（前16个通道）
conv3_maps = activations['conv3_out'][0]  # (128, 3, 3)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat[:16]):
    ax.imshow(conv3_maps[i].cpu(), cmap='viridis', interpolation='nearest')
    ax.set_title(f'Channel {i}')
    ax.axis('off')
plt.suptitle(f'Conv3 特征图 (空间尺寸仅3×3) - 高级特征：物体部件', fontsize=14)
plt.tight_layout()
plt.show()

print("\n=== 特征图观察总结 ===")
print("Conv1 (28×28): 图像细节完整，检测低级的边缘和颜色变化")
print("Conv2 (14×14): 空间缩小一半，开始出现中层模式（曲线、拐角、纹理组合）")
print("Conv3 (3×3):   空间压缩到3×3，每个像素编码了较大感受野的高级语义")
print(""这是CNN的核心机制：空间分辨率逐步降低，语义抽象级别逐步提高"")</cell>


## 4.7 著名CNN架构演进史

### LeNet-5 (1998) — 开山鼻祖
- 作者：Yann LeCun
- 结构：Conv→Pool→Conv→Pool→FC→FC
- 贡献：首次将反向传播用于CNN，MNIST手写数字识别
- 参数：约6万，非常轻量

### AlexNet (2012) — 深度学习爆发
- ImageNet 2012冠军，将错误率从26%降到15%
- 关键创新：
  - **ReLU激活函数**：解决sigmoid的梯度消失
  - **Dropout**：防止过拟合
  - **GPU训练**：开创性地用双GPU并行
  - 更大的规模：5个Conv层 + 3个FC层, 6000万参数
- 11×11大卷积核（现代CNN很少用这么大的）

### VGG (2014) — 深度与简洁
- 完全用3×3小卷积核堆叠（非常规则的设计）
- 两个3×3卷积 ≈ 一个5×5的感受野（但参数更少，非线性更多）
- VGG16: 16层, 1.38亿参数
- VGG19: 19层
- 问题：参数太多（FC层占绝大多数），推理慢

### GoogLeNet/Inception (2014) — 宽度与效率
- Inception模块：并行使用1×1、3×3、5×5多种卷积核
- 1×1卷积降低通道数，大幅减少计算量
- 无全连接层（用全局平均池化替代）
- 参数仅约500万——比VGG的1.38亿少了96%

### ResNet (2015) — 残差学习
- **核心创新：残差连接（skip connection）**
$$y = \mathcal{F}(x) + x$$

- 解决了什么？**深度增加 = 性能退化**的悖论
  - 之前：越深的网络训练越困难（梯度消失/爆炸）
  - ResNet：152层的网络比更浅的VGG表现更好！
- 为什么有效？
  - 恒等映射路径让梯度可以直接回流
  - 网络学习的是"残差" $\mathcal{F}(x) = H(x) - x$，而非直接学习$H(x)$
  - 即使中间层学不到任何东西，至少可以走"短路"（$\mathcal{F}(x)=0, y=x$）

```
传统网络:              ResNet:
x → [Conv→BN→ReLU]    x → [Conv→BN→ReLU]
  → [Conv→BN→ReLU]      → [Conv→BN] → + ← x (skip!)
  → H(x)                  → ReLU → y = F(x) + x
```

### 架构演进的核心趋势

| 网络 | 年份 | 深度 | 核心创新 | 参数量 |
|------|------|------|----------|--------|
| LeNet-5 | 1998 | 5层 | CNN原型 | ~60K |
| AlexNet | 2012 | 8层 | ReLU+Dropout+GPU | ~60M |
| VGG16 | 2014 | 16层 | 全是3×3小卷积 | ~138M |
| Inception v1 | 2014 | 22层 | 多尺度+1×1降维 | ~5M |
| ResNet-50 | 2015 | 50层 | 残差连接 | ~25M |
| ResNet-152 | 2015 | 152层 | 残差连接 | ~60M |
| EfficientNet | 2019 | 不同 | 复合缩放(depth+width+resolution) | ~5-66M |

> **趋势：更深的网络需要残差连接；更高效的设计（1×1卷积、深度可分离卷积）追求参数/性能比。ResNet的"残差思想"影响了后续几乎所有架构，包括Transformer！**</cell>


In [ ]:
import torch
import torch.nn as nn

# ============================================================
# 手动实现ResNet的残差块（Residual Block）
# ============================================================
class ResidualBlock(nn.Module):
    """ResNet的基本构建块：两个3×3卷积 + 残差连接"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 如果输入输出维度不匹配（通道数或空间尺寸），需要1×1卷积调整
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = x  # 保存原始输入（短路连接）
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = nn.ReLU(inplace=True)(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 核心：逐元素加法！out = F(x) + x
        out += self.shortcut(residual)
        out = nn.ReLU(inplace=True)(out)
        
        return out

# 测试残差块
block = ResidualBlock(64, 64, stride=1)
x = torch.randn(4, 64, 32, 32)
out = block(x)
print(f"残差块 (通道不変): {x.shape} → {out.shape}")
assert x.shape == out.shape, "尺寸应该不变"

# 下采样残差块（空间减半，通道加倍）
block_down = ResidualBlock(64, 128, stride=2)
out_down = block_down(x)
print(f"残差块 (下采样): {x.shape} → {out_down.shape}")

# 解释残差连接为何有助于梯度流动
print(f"\n=== 残差连接的梯度优势 ===")
print("标准网络:  梯度 ∝ ∏ prod(W_i) → 连乘, 容易消失/爆炸")
print("残差网络:  梯度 ∝ ∏ prod(W_i) + 1 → 至少有'1'这个常数梯度通路")
print("即使所有W_i都≈0，梯度也能通过'1'的路径回流，深层的参数仍然能学到东西")
print("这就是ResNet能训练152层而VGG到19层就难以继续加深的根本原因！")</cell>


In [ ]:
import torch
import torch.nn as nn

# ============================================================
# 1×1卷积详解：为什么它能降维且不损失空间信息？
# ============================================================

# 1×1卷积看起来像"逐通道的线性组合"
# 输入: (B, C_in, H, W)
# 权重: (C_out, C_in, 1, 1) — 空间维度=1
# 输出: (B, C_out, H, W) — 空间尺寸不变！

# 示例：用1×1卷积将128通道降到32通道
conv_1x1 = nn.Conv2d(128, 32, kernel_size=1)
x = torch.randn(16, 128, 14, 14)
out = conv_1x1(x)
print(f"1×1卷积: {x.shape} → {out.shape}")
print(f"参数: {sum(p.numel() for p in conv_1x1.parameters()):,}  (= 128×32+32)")

# 对比：3×3卷积
conv_3x3 = nn.Conv2d(128, 32, kernel_size=3, padding=1)
out_3x3 = conv_3x3(x)
print(f"\n3×3卷积: {x.shape} → {out_3x3.shape}")
print(f"参数: {sum(p.numel() for p in conv_3x3.parameters()):,}  (= 3×3×128×32+32)")
print(f"计算量: {3*3*128*32*14*14*16:,} 次乘加运算")

# 1×1卷积的计算量
flops_1x1 = 1*1*128*32*14*14*16
print(f"1×1计算量: {flops_1x1:,} 次乘加运算")
print(f"节省倍数: {3*3 / (1*1):.0f}× (正好是卷积核面积比)")

print(f"\n=== 1×1卷积的多重角色 ===")
print("1. 降维压缩: 减少通道数 → 降低后续3×3卷积的计算量")
print("2. 升维扩展: 增加通道数 → 增强表示能力")
print("3. 跨通道信息融合: 每个输出通道是所有输入通道的线性组合")
print("4. 非线性引入: 配合ReLU → 增加决策边界的灵活性")
print("5. 参数量: C_in × C_out, 极其高效")</cell>


## 4.8 深度可分离卷积与MobileNet

### 为什么还需要进一步减少计算量？

ResNet虽好，但25M参数在移动设备上仍然太大。**深度可分离卷积(Depthwise Separable Convolution)**将标准卷积分解为两步：

**标准卷积**（一次完成空间+通道混合）：
$$P = K^2 \times C_{in} \times C_{out}$$

**深度可分离卷积**（分两步）：

**Step 1 — 逐通道卷积(Depthwise)**：每个输入通道用自己专属的K×K核
$$P_{dw} = K^2 \times C_{in}$$

**Step 2 — 逐点卷积(Pointwise)**：用1×1卷积混合通道信息
$$P_{pw} = C_{in} \times C_{out}$$

**总参数**：$K^2 \times C_{in} + C_{in} \times C_{out}$

### 参数减少比例

$$\frac{P_{ds}}{P_{std}} = \frac{K^2 C_{in} + C_{in} C_{out}}{K^2 C_{in} C_{out}} = \frac{1}{C_{out}} + \frac{1}{K^2}$$

当$C_{out}=64, K=3$时：
$$\frac{1}{64} + \frac{1}{9} \approx 0.127$$

**参数减少约87%！** 这就是MobileNet的核心思想。

### MobileNet系列演进
| 版本 | 创新 | 参数量 |
|------|------|--------|
| MobileNetV1 (2017) | 深度可分离卷积 | ~4.2M |
| MobileNetV2 (2018) | 倒残差结构+线性瓶颈 | ~3.5M |
| MobileNetV3 (2019) | NAS搜索+SE注意力 | ~5.4M |

> **启示：** 从标准卷积→深度可分离卷积→1×1卷积的各种组合，背后是同一个问题：**如何在保持表达能力的同时，尽可能减少计算量和参数量？** 这体现了CNN设计的核心权衡。</cell>


In [ ]:
import torch
import torch.nn as nn

# ============================================================
# 深度可分离卷积实现
# ============================================================
class DepthwiseSeparableConv(nn.Module):
    """深度可分离卷积 = Depthwise + Pointwise"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        # Step 1: 逐通道卷积 — 每个输入通道单独做空间卷积
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size,
                                    stride=stride, padding=padding, groups=in_channels)
        # Step 2: 逐点卷积 — 1×1卷积混合通道
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    
    def forward(self, x):
        return self.pointwise(self.depthwise(x))

# 对比标准卷积和深度可分离卷积
in_c, out_c, k = 32, 64, 3
std_conv = nn.Conv2d(in_c, out_c, k, padding=1, bias=False)
ds_conv = DepthwiseSeparableConv(in_c, out_c, k)

std_params = sum(p.numel() for p in std_conv.parameters())
ds_params = sum(p.numel() for p in ds_conv.parameters())

print(f"标准卷积参数: {std_params:,}")
print(f"深度可分离卷积参数: {ds_params:,}")
print(f"参数减少: {(1 - ds_params/std_params)*100:.1f}%")
print(f"理论比例: {(1/out_c + 1/k**2)*100:.1f}% (= 1/C_out + 1/K²)")

# 验证输出形状一致
x = torch.randn(4, in_c, 32, 32)
out_std = std_conv(x)
out_ds = ds_conv(x)
print(f"\n标准卷积输出: {out_std.shape}")
print(f"深度可分离卷积输出: {out_ds.shape}")
print(f"形状一致: {out_std.shape == out_ds.shape}")</cell>


## 4.4 观察3 → 简化3：汇聚

### 观察
把图片缩小一半，我们依然能认出里面的内容。**子采样不影响语义理解。**

### 简化：汇聚/池化 (Pooling)
定期对特征图进行下采样，减小空间尺寸：
- **最大汇聚(Max Pooling)**：取区域内最大值 → 保留最强特征
- **平均汇聚(Avg Pooling)**：取区域内平均值 → 更平滑

汇聚的好处：
1. 降低特征图尺寸 → 后续层计算量指数级减少
2. 扩大感受野（同样的3×3卷积在缩小后的图上覆盖更大的原图区域）
3. 提供一定程度的平移不变性

### CNN标准组件
```
Conv2d → BatchNorm → ReLU → MaxPool  ← 一个标准卷积块
```
重复堆叠 → 空间尺寸越来越小，通道数越来越多 → 最终通过全连接层分类

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 一个标准的CNN（类LeNet风格）
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1: 1→32通道
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        # Block 2: 32→64通道
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        # 分类头
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # 28→14
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # 14→7
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = CNN()
x = torch.randn(4, 1, 28, 28)
out = model(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}")

# 统计参数量
print(f"\n参数统计：")
for name, param in model.named_parameters():
    print(f"  {name}: {param.numel():,} params")
total = sum(p.numel() for p in model.parameters())
print(f"  总计: {total:,} 参数")

# 对比：如果第一层是全连接
fc_params = 28*28 * 32*28*28  # 全连接的参数（夸张）
conv_params = 1*32*3*3  # 卷积的参数
print(f"\nConv2d(1→32,3x3) 参数: {conv_params}")
print(f"相同输入输出若用全连接: ≈{fc_params:,} 参数 (爆炸！)")

## 4.5 感受野 + 共享参数 = 卷积层

综合两个简化，就得到了**卷积层(Convolutional Layer)**：

$$\text{输出}[i,j,k] = \sum_{c=0}^{C_{in}-1} \sum_{u=0}^{K-1} \sum_{v=0}^{K-1} \text{输入}[i+u, j+v, c] \cdot \text{权重}_k[u, v, c] + \text{偏置}_k$$

其中$k$是输出通道（滤波器）的索引，$C_{in}$是输入通道数，$K$是卷积核大小。

### 标准卷积参数量
$$P_{\text{conv}} = K \times K \times C_{in} \times C_{out} + C_{out}$$

例如：3×3核，输入32通道，输出64通道 → $3^2 \times 32 \times 64 + 64 = 18,496$ 参数

### 核心收获
> **CNN的本质** = 带空间约束的全连接网络。两个约束是：(1)局部连接(感受野)，(2)权重共享。这两个约束来自对图像的"先验知识"——它们不是限制，而是**归纳偏置(inductive bias)**，让模型用更少的参数学到更好的特征。

## 4.6 CNN在围棋中的应用 (AlphaGo)

围棋棋盘19×19，也是一个"网格状"输入。CNN同样适用：
- **局部性**：棋子只影响周围区域
- **平移不变性**：同样的棋型在棋盘任何位置含义相同

**但AlphaGo去掉了汇聚层(Pooling)！**

为什么？围棋需要**精确的位置信息**——一个子在(3,3)和(3,4)天差地别。Pooling的下采样会丢失这种精确性。

> 这说明CNN三大设计并非一定要全部使用——要根据具体问题灵活调整。

## 本章核心收获
1. CNN = 感受野(局部连接) + 共享参数(卷积核) + 汇聚(下采样)
2. 三大设计都来自对图像的观察，每一项都有明确动机
3. 卷积层参数量 = K²×C_in×C_out，远小于全连接
4. 平移等变性是CNN的核心归纳偏置
5. 汇聚层可选——需要精确位置时可以不���
6. CNN适用于任何有空间\网格结构的输入（图像、棋盘、音频频谱图）

## 4.9 本章知识总结与思考题

### 核心知识回顾

1. **CNN = 三个简化**：感受野(局部连接) + 参数共享(卷积核) + 汇聚(下采样)
2. **卷积操作** = 核在输入上滑动，逐个位置计算元素乘积之和
3. **输出尺寸公式**：$W_{out} = \lfloor(W_{in} - K + 2P)/S\rfloor + 1$
4. **参数量**：$K^2 \times C_{in} \times C_{out}$ — 与输入空间尺寸无关！
5. **1×1卷积** = 跨通道线性组合，保持空间尺寸，用于降维/升维
6. **深度可分离卷积** = Depthwise + Pointwise，参数减少 ~87%
7. **残差连接**：$y = F(x) + x$，解决深度网络的梯度退化问题
8. **架构演进**：LeNet → AlexNet → VGG → Inception → ResNet → MobileNet → EfficientNet

### 思考题

1. 为什么两个3×3卷积的感受野等于一个5×5卷积？参数有什么差异？
2. BatchNorm在训练和推理时的行为有什么不同？（提示：running_mean）
3. 全局平均池化(Global Avg Pooling)为什么能替代全连接分类层？
4. 如果输入是256×256而非28×28，CNN的参数数量会改变吗？全连接呢？
5. ResNet的残差连接为什么对"网络加深性能不降反升"是关键？
6. 为什么AlphaGo移除Pooling层——这反映了CNN设计的什么灵活性？

### 延伸阅读
- "A guide to convolution arithmetic for deep learning" (Dumoulin & Visin, 2018) — 卷积运算动画详解
- "Visualizing and Understanding Convolutional Networks" (Zeiler & Fergus, 2014) — 反卷积可视化
- "Deep Residual Learning for Image Recognition" (He et al., 2015) — ResNet原论文
- "MobileNets: Efficient Convolutional Neural Networks" (Howard et al., 2017) — 移动端CNN</cell>
